In [ ]:
# 12_buffer_parallel_trends.ipynb
# Revision robustness for the insurance BUFFER. (a) Treated-sample composition and a
# test of whether the shock induces panel attrition. (b) Two-way FE event study with
# event-time x baseline-PHI interactions (placebo pre-trends for the buffer).
# (c) Callaway-Sant'Anna event study estimated SEPARATELY within each baseline-PHI
# group, and balanced-core re-estimation. Writes t12_*, t13_* and fig15 (grayscale).
# Experiments A (buffer parallel-trends / placebo) and B (treated N, attrition, balanced core)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from linearmodels.panel import PanelOLS
from differences import ATTgt

PROC="../data/processed"; TAB="../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet").sort_values(["PIDWON","year"]).reset_index(drop=True)
p["che40f"]=p["che40"].astype(float); p["che25f"]=p["che25"].astype(float)

# ================= Experiment B: treated N and composition =================
tc=p.groupby("PIDWON")["cohort"].first()
bp=p.groupby("PIDWON")["base_phi"].first()
treated=tc[tc>0].index
print("=== Treated composition (acute shock = first new hospitalization) ===")
print("treated persons:", (tc>0).sum(), "| never-treated:", (tc==0).sum())
print("treated & baseline-UNINSURED (DBCR target):", int(((tc>0)&(bp==0)).sum()))
print("treated & baseline-INSURED:", int(((tc>0)&(bp==1)).sum()))
nyr=p.groupby("PIDWON")["year"].nunique()
print("treated persons in balanced 6y core:", int(((tc>0)&(nyr==6)).sum()))

# ---- Attrition test: does the shock predict leaving the panel next wave? ----
p=p.sort_values(["PIDWON","year"]);
p["nextobs"]=p.groupby("PIDWON")["year"].shift(-1)
p["exit_next"]=((p["year"]<2024) & (p["nextobs"].isna())).astype(float)
p.loc[p["year"]==2024,"exit_next"]=np.nan
# post-shock indicator on the acute shock
p["post_ac"]=((p["cohort"]>0)&(p["year"]>=p["cohort"])).astype(int)
d=p[p["year"]<2024].set_index(["PIDWON","year"])
att_e=PanelOLS(d["exit_next"], d[["post_ac"]], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
print("\n=== Attrition: exit_next ~ post_shock (person+year FE) ===")
print("coef=%.4f  se=%.4f  p=%.3f  (positive => shock-related attrition)"%(att_e.params["post_ac"],att_e.std_errors["post_ac"],att_e.pvalues["post_ac"]))
print("raw attrition rate: post-shock person-years=%.3f  others=%.3f"%(
    p.loc[p["post_ac"]==1,"exit_next"].mean(), p.loc[p["post_ac"]==0,"exit_next"].mean()))

# ================= Experiment A: event study by baseline PHI =================
# interacted TWFE event study: e_k and e_k*base_phi; pre-period interactions test
# whether insured vs uninsured were on PARALLEL pre-shock paths (the buffer's identifying assumption)
p["k"]=np.where(p["cohort"]>0, p["year"]-p["cohort"], np.nan)
kb=p["k"].apply(lambda k: np.nan if pd.isna(k) else max(-4,min(4,int(k))))
p["kb"]=kb
evk=[-4,-3,-2,0,1,2,3,4]
maincols=[]; intcols=[]
for kk in evk:
    p[f"e{kk}"]=((p["cohort"]>0)&(p["kb"]==kk)).astype(int)
    p[f"e{kk}_phi"]=p[f"e{kk}"]*p["base_phi"]
    maincols.append(f"e{kk}"); intcols.append(f"e{kk}_phi")
cnt=p.groupby("PIDWON")["year"].transform("size")
es=p[cnt>=2].set_index(["PIDWON","year"])
d=es[["che40f"]+maincols+intcols].dropna()
m=PanelOLS(d["che40f"], d[maincols+intcols], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
rows=[]
for kk in evk:
    rows.append({"k":kk,
        "uninsured":m.params[f"e{kk}"],"uninsured_se":m.std_errors[f"e{kk}"],
        "insured_diff":m.params[f"e{kk}_phi"],"insured_diff_se":m.std_errors[f"e{kk}_phi"],
        "insured_diff_p":m.pvalues[f"e{kk}_phi"]})
es_phi=pd.DataFrame(rows)
es_phi.to_csv(f"{TAB}/t12_buffer_eventstudy_byphi.csv", index=False)
print("\n=== Event study by baseline PHI (CHE40): interaction = insured-minus-uninsured path ===")
print(es_phi.round(4).to_string(index=False))
# joint Wald test of PRE-period interaction terms (k=-4,-3,-2): parallel pre-trends for the buffer.
# (build a restriction matrix R by name to avoid formula parsing of the '-' in variable names)
def joint_wald(res, names):
    idx=list(res.params.index); R=np.zeros((len(names),len(idx)))
    for i,nm in enumerate(names): R[i, idx.index(nm)]=1.0
    return res.wald_test(restriction=R, value=np.zeros(len(names)))
pre_int=[f"e{kk}_phi" for kk in [-4,-3,-2]]
wald=joint_wald(m, pre_int)
print("\nJoint test of PRE-period interaction coefficients (k=-4,-3,-2) = 0:")
print("  stat=%.3f  p=%.4f  (p>0.05 => insured/uninsured parallel before the shock)"%(wald.stat, wald.pval))
pre_main=[f"e{kk}" for kk in [-4,-3,-2]]
wald2=joint_wald(m, pre_main)
print("Joint test of PRE-period main (uninsured) coefficients = 0: stat=%.3f p=%.4f"%(wald2.stat,wald2.pval))


# ---- (c) CS event study by PHI subgroup + balanced core + Figure 15 ----
# Experiment A2: Callaway-Sant'Anna event study (pre-trends) SEPARATELY by baseline PHI,
# and balanced-core re-estimation of overall ATT and buffer.
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from differences import ATTgt
PROC="../data/processed"; TAB="../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet")
p["che40f"]=p["che40"].astype(float)
nyr=p.groupby("PIDWON")["year"].transform("nunique")
p["balanced6"]=(nyr==6).astype(int)

def cs_event(df, outcome="che40f"):
    d=df[["PIDWON","year","cohort",outcome]].dropna(subset=[outcome]).copy()
    d["cohort"]=d["cohort"].replace(0,np.nan)
    d=d.set_index(["PIDWON","year"])
    att=ATTgt(data=d, cohort_column="cohort", base_period="universal")
    att.fit(formula=outcome, est_method="dr", control_group="not_yet_treated", progress_bar=False)
    e=att.aggregate("event"); e.columns=[str(c) for c in e.columns]
    s=att.aggregate("simple"); s.columns=[str(c) for c in s.columns]
    ev={int(rp):(float(r.iloc[0]), float(r.iloc[1]) if not pd.isna(r.iloc[1]) else np.nan) for rp,r in e.iterrows()}
    return float(s.iloc[0,0]), float(s.iloc[0,1]), ev

print("=== CS event study (CHE40) pre-trends by baseline PHI subgroup ===")
rows=[]
for grp,lab in [(0,"No PHI"),(1,"Has PHI")]:
    simple,se,ev=cs_event(p[p["base_phi"]==grp])
    print(f"\n[{lab}] overall ATT=%.4f (se %.4f)"%(simple,se))
    for k in sorted(ev):
        rows.append({"group":lab,"k":k,"ATT":ev[k][0],"se":ev[k][1]})
        print(f"  k={k:+d}  ATT=%.4f  se=%.4f"%(ev[k][0],ev[k][1]))
pd.DataFrame(rows).to_csv(f"{TAB}/t12_cs_eventstudy_byphi.csv", index=False)

print("\n=== Balanced-core (6-year persons) re-estimation ===")
pb=p[p["balanced6"]==1]
s0,se0,_=cs_event(pb[pb["base_phi"]==0]); s1,se1,_=cs_event(pb[pb["base_phi"]==1])
sa,sea,_=cs_event(pb)
print("balanced overall ATT CHE40=%.4f (se %.4f)"%(sa,sea))
print("balanced buffer: No PHI=%.4f (se %.4f) | Has PHI=%.4f (se %.4f)"%(s0,se0,s1,se1))
pd.DataFrame([{"sample":"balanced6","overall":sa,"overall_se":sea,
               "noPHI":s0,"noPHI_se":se0,"hasPHI":s1,"hasPHI_se":se1}]).to_csv(f"{TAB}/t13_balanced_core.csv", index=False)


# ---- Figure 15: CS event study CHE40 by baseline PHI ----
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size":11,"axes.edgecolor":"0.3","savefig.dpi":600,"figure.dpi":110,"grid.color":"0.9"})
es=pd.read_csv(f"{TAB}/t12_cs_eventstudy_byphi.csv")
fig,ax=plt.subplots(figsize=(7.2,4.4))
for grp,fmt,shade,mf in [("No PHI","o-","0.12","white"),("Has PHI","s--","0.55","0.55")]:
    g=es[es["group"]==grp].sort_values("k")
    ax.errorbar(g["k"],g["ATT"],yerr=1.96*g["se"],fmt=fmt,color=shade,ms=6,lw=1.7,capsize=3,
                ecolor="0.5",markerfacecolor=mf,markeredgewidth=1.3,label=grp)
ax.axhline(0,color="0.5",lw=0.8); ax.axvline(-0.5,color="0.7",ls="--",lw=1)
ax.set_xlabel("Years relative to first hospitalization (k)"); ax.set_ylabel("ATT on P(CHE 40%) vs k=-1 (Callaway-Sant'Anna)")
ax.set_xticks(sorted(es["k"].unique())); ax.legend(frameon=False,title="Baseline insurance")
for ext in ("png","pdf"): fig.savefig(f"../results/figures/fig15_buffer_eventstudy_phi.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig); print("fig15 written")
